<a href="https://colab.research.google.com/github/paridhietal/TransferLearning/blob/main/transferLearning3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_memory")
collection = client.get_or_create_collection("agent_memory")

In [ ]:
def retrieve_relevant_memory(query, top_k=3):
  results = collection.query(
      query_texts=[query],
      n_results=top_k,
  )
  hits = []
  for i in range(len(results['ids'][0])):
    hit = {
        "id": results['ids'][0][i],
        "document": results['documents'][0][i],
        "task": results['metadatas'][0][i]['task'],
        "lesson": results['metadatas'][0][i]['lesson'],
        "score": results['metadatas'][0][i]['score']
    }
    hits.append(hit)
  return hits

def build_prompt_with_memory(new_query):
  hits = retrieve_relevant_memory(new_query, top_k=3)

  #Build memory context block
  context = ""
  for i, h in enumerate(hits):
    context += f"""
Past Example {i+1}:
  Task: {h['task']}
  Lesson: {h['lesson']}
  Score: {h['score']}
"""
  #Final Prompt
  prompt = f"""You are an intelligent agent with memory of past work.

Here are the most relevant past experiences for this topic:
{context}

Now use those lessons to respond to this new task:
{new_query}

Give a detailed, improved response based on what was learned before."""

  return prompt, hits

In [ ]:
def run_with_memory(new_query, llm_fn):
  prompt, hits = build_prompt_with_memory(new_query)

  print("===Relevant Memories Found===")
  for h in hits:
    print(f" -{h['task']} (score={h['score']})")

  print("\n=== Generating Response===")
  output = llm_fn(prompt)
  print(output)

  return prompt, output, hits

In [ ]:
def save_new_memory(new_query, output, score, lesson):
  new_id = str(collection.count() + 1)

  collection.add(
      ids=[new_id],
      documents=[f"{new_query} {lesson}"],
      metadatas=[{
          "task": new_query,
          "output": output,
          "lesson": lesson,
          "score": str(score)
      }]
  )

  # client.persist() # Removed as PersistentClient automatically persists.
  print(f"New memory saved. Total: {collection.count()} entries")

In [ ]:
#This is your main entry point from now on
query = "your new topic here"

# Define your LLM function here. For example, using a placeholder:
def your_llm_function(prompt):
    # Replace this with your actual LLM call (e.g., from an API or a local model)
    return f"LLM response to: {prompt}"

prompt, output, hits = run_with_memory(query, your_llm_function)

#After reviewing the output, save it back
save_new_memory(
    new_query=query,
    output=output,
    score=8,
    lesson="what worked this time"
)

===Relevant Memories Found===
 -your new topic here (score=8)

=== Generating Response===
LLM response to: You are an intelligent agent with memory of past work.

Here are the most relevant past experiences for this topic:

Past Example 1:
  Task: your new topic here
  Lesson: what worked this time
  Score: 8


Now use those lessons to respond to this new task:
your new topic here

Give a detailed, improved response based on what was learned before.
New memory saved. Total: 2 entries
